# Sajag V2 — Custom Multilingual Scam Classifier (MuRIL -> INT8 ONNX)

Run on **Google Colab Free Tier (T4 GPU)**.

**Inputs required** (upload from `backend/data/ml_dataset/` or mount Google Drive):
- `train.csv`, `val.csv`, `test.csv` (columns: `text`, `label`, `language`)

**Gate C Exit Criteria**:
- Test scam-class F1 >= `0.87`
- Every language (`English`, `Hindi`, `Hinglish`, `Marathi`) F1 >= `0.80`
- Quantized ONNX (`model_quantized.onnx`) < `150 MB`
- PyTorch vs ONNX probability parity within `1e-2` on 50 test samples
- Exported `model_quantized.onnx`, `tokenizer.json`, `config.json` ready for Hugging Face Hub


In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn onnx onnxruntime onnxscript huggingface_hub seaborn matplotlib

In [ ]:
import inspect
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset, DatasetDict
from onnxruntime.quantization import QuantType, quantize_dynamic
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from tokenizers import Tokenizer
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

SEED = 42
MAX_LENGTH = 128
MODEL_ID = "google/muril-base-cased"
# Fallback if Gate C F1 fails after tuning or quantized ONNX >= 150 MB:
# MODEL_ID = "distilbert-base-multilingual-cased"
SPECIAL_TOKENS = ["[URL]", "[PHONE]"]
LANGUAGES = ["English", "Hindi", "Hinglish", "Marathi"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load Prepared Dataset Splits
Upload `train.csv`, `val.csv`, `test.csv` from `backend/data/ml_dataset/` into `/content/ml_dataset/` (or current working directory).


In [ ]:
DATA_DIR = Path("/content/ml_dataset") if Path("/content/ml_dataset/train.csv").exists() else Path(".")
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert list(df.columns) == ["text", "label", "language"], f"Unexpected columns in {name}: {df.columns}"
    scam_pct = df["label"].mean() * 100
    print(f"{name:5s}: {len(df):4d} rows | scam={scam_pct:.1f}% | langs={df['language'].value_counts().to_dict()}")

## 2. Tokenization & Special Tokens (`[URL]`, `[PHONE]`)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
num_added = tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
print(f"Added {num_added} special tokens. Total vocab size: {len(tokenizer)}")

for tok in SPECIAL_TOKENS:
    tok_ids = tokenizer.encode(tok, add_special_tokens=False)
    assert len(tok_ids) == 1, f"Expected single token for {tok}, got {tok_ids}"
    print(f"  {tok} -> token_id={tok_ids[0]}")

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label"]]),
    "validation": Dataset.from_pandas(val_df[["text", "label"]]),
    "test": Dataset.from_pandas(test_df[["text", "label"]]),
})

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

tokenized = raw_datasets.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized = tokenized.rename_column("label", "labels")
cols = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in tokenized["train"].column_names:
    cols.append("token_type_ids")
tokenized.set_format("torch", columns=cols)
print("Tokenized columns:", cols)

## 3. Model Setup & Training (with `resize_token_embeddings` and Early Stopping)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: "ham", 1: "scam"},
    label2id={"ham": 0, "scam": 1},
)
model.resize_token_embeddings(len(tokenizer))

scam_ratio = float(train_df["label"].mean())
use_class_weights = scam_ratio > 0.55 or scam_ratio < 0.45
class_weights_tensor = None
if use_class_weights:
    weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df["label"].values)
    class_weights_tensor = torch.tensor(weights, dtype=torch.float32)
    print("Applying class weights:", weights.tolist())
else:
    print(f"Balanced train set ({scam_ratio * 100:.1f}% scam); using unweighted loss.")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weight = class_weights_tensor.to(logits.device) if class_weights_tensor is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1": float(f1_score(labels, preds, pos_label=1)),
        "precision": float(precision_score(labels, preds, pos_label=1, zero_division=0)),
        "recall": float(recall_score(labels, preds, pos_label=1, zero_division=0)),
    }

training_args = TrainingArguments(
    output_dir="./muril_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=25,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 4. Save Best Model & Evaluate Gate C Metrics


In [ ]:
FINAL_DIR = Path("./muril_scam_final")
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

pred_out = trainer.predict(tokenized["test"])
y_logits = pred_out.predictions
y_probs = torch.softmax(torch.tensor(y_logits), dim=-1).numpy()
y_pred = np.argmax(y_probs, axis=-1)
y_true = pred_out.label_ids

overall_metrics = {
    "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    "f1": round(float(f1_score(y_true, y_pred, pos_label=1)), 4),
    "precision": round(float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)), 4),
    "recall": round(float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)), 4),
}
print("Overall test metrics:", overall_metrics)
print(classification_report(y_true, y_pred, target_names=["ham", "scam"], digits=4))

per_lang_metrics = {}
test_eval_df = test_df.copy()
test_eval_df["pred"] = y_pred
for lang in LANGUAGES:
    sub = test_eval_df[test_eval_df["language"] == lang]
    lang_f1 = round(float(f1_score(sub["label"], sub["pred"], pos_label=1, zero_division=0)), 4)
    lang_acc = round(float(accuracy_score(sub["label"], sub["pred"])), 4)
    per_lang_metrics[lang] = {"samples": int(len(sub)), "accuracy": lang_acc, "f1": lang_f1}
    print(f"=== {lang} ({len(sub)} samples) | acc={lang_acc:.4f} | f1={lang_f1:.4f} ===")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["ham", "scam"], yticklabels=["ham", "scam"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Sajag Classifier - Test Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

assert overall_metrics["f1"] >= 0.87, f"Gate C FAILED: test F1 {overall_metrics['f1']} < 0.87"
for lang, stats in per_lang_metrics.items():
    assert stats["f1"] >= 0.80, f"Gate C FAILED: {lang} F1 {stats['f1']} < 0.80"
print("Gate C accuracy & per-language F1 checks PASSED.")

## 5. Export ONNX (Opset 17), Quantize INT8, and Save `tokenizer.json`


In [ ]:
ONNX_DIR = Path("./muril_onnx")
ONNX_DIR.mkdir(parents=True, exist_ok=True)
FP32_ONNX = ONNX_DIR / "model.onnx"
QUANT_ONNX = ONNX_DIR / "model_quantized.onnx"

cpu_model = AutoModelForSequenceClassification.from_pretrained(str(FINAL_DIR)).cpu().eval()
forward_params = list(inspect.signature(cpu_model.forward).parameters.keys())
uses_token_type_ids = "token_type_ids" in forward_params

sample_enc = tokenizer(
    ["sample text"],
    truncation=True,
    max_length=MAX_LENGTH,
    padding="max_length",
    return_tensors="pt",
)
input_names = ["input_ids", "attention_mask"]
dummy_inputs = (sample_enc["input_ids"], sample_enc["attention_mask"])
dynamic_axes = {
    "input_ids": {0: "batch_size", 1: "sequence"},
    "attention_mask": {0: "batch_size", 1: "sequence"},
    "logits": {0: "batch_size"},
}
if uses_token_type_ids:
    input_names.append("token_type_ids")
    dummy_inputs = (*dummy_inputs, sample_enc["token_type_ids"])
    dynamic_axes["token_type_ids"] = {0: "batch_size", 1: "sequence"}

export_kwargs = dict(
    f=str(FP32_ONNX),
    input_names=input_names,
    output_names=["logits"],
    dynamic_axes=dynamic_axes,
    opset_version=17,
    do_constant_folding=True,
)
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    export_kwargs["dynamo"] = False

torch.onnx.export(cpu_model, dummy_inputs, **export_kwargs)

quantize_dynamic(
    model_input=str(FP32_ONNX),
    model_output=str(QUANT_ONNX),
    weight_type=QuantType.QUInt8,
)

tokenizer.save_pretrained(str(ONNX_DIR))
runtime_config = {
    "model_id": MODEL_ID,
    "max_length": MAX_LENGTH,
    "pad_token_id": int(tokenizer.pad_token_id),
    "id2label": {"0": "ham", "1": "scam"},
    "label2id": {"ham": 0, "scam": 1},
    "special_tokens": SPECIAL_TOKENS,
    "uses_token_type_ids": uses_token_type_ids,
}
(ONNX_DIR / "config.json").write_text(json.dumps(runtime_config, indent=2), encoding="utf-8")

fp32_mb = FP32_ONNX.stat().st_size / (1024 * 1024)
quant_mb = QUANT_ONNX.stat().st_size / (1024 * 1024)
print(f"FP32 ONNX: {fp32_mb:.2f} MB | Quantized INT8 ONNX: {quant_mb:.2f} MB")
assert (ONNX_DIR / "tokenizer.json").exists(), "tokenizer.json missing!"
assert quant_mb < 150.0, f"Gate C FAILED: quantized ONNX {quant_mb:.2f} MB >= 150 MB"

metrics_payload = {
    "model_id": MODEL_ID,
    "overall": overall_metrics,
    "per_language": per_lang_metrics,
    "confusion_matrix": cm.tolist(),
    "fp32_onnx_mb": round(fp32_mb, 2),
    "quantized_onnx_mb": round(quant_mb, 2),
}
(ONNX_DIR / "test_metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

## 6. Probability Parity Check (PyTorch vs FP32 ONNX vs INT8 ONNX via `tokenizers.Tokenizer`)


In [ ]:
sample_texts = test_df["text"].head(50).tolist()

# PyTorch probabilities
pt_enc = tokenizer(
    sample_texts,
    truncation=True,
    max_length=MAX_LENGTH,
    padding="max_length",
    return_tensors="pt",
)
with torch.no_grad():
    pt_kwargs = {"input_ids": pt_enc["input_ids"], "attention_mask": pt_enc["attention_mask"]}
    if uses_token_type_ids:
        pt_kwargs["token_type_ids"] = pt_enc["token_type_ids"]
    pt_probs = torch.softmax(cpu_model(**pt_kwargs).logits, dim=-1).numpy()

# Runtime tokenizers + ONNX check (exact production path D3)
fast_tok = Tokenizer.from_file(str(ONNX_DIR / "tokenizer.json"))
fast_tok.enable_truncation(max_length=MAX_LENGTH)
fast_tok.enable_padding(pad_id=int(tokenizer.pad_token_id), length=MAX_LENGTH)
encoded = fast_tok.encode_batch(sample_texts)

ort_inputs = {
    "input_ids": np.array([e.ids for e in encoded], dtype=np.int64),
    "attention_mask": np.array([e.attention_mask for e in encoded], dtype=np.int64),
}
if uses_token_type_ids:
    ort_inputs["token_type_ids"] = np.array([e.type_ids for e in encoded], dtype=np.int64)

def softmax_np(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)

fp32_sess = ort.InferenceSession(str(FP32_ONNX), providers=["CPUExecutionProvider"])
fp32_probs = softmax_np(fp32_sess.run(None, ort_inputs)[0])
fp32_diff = float(np.max(np.abs(pt_probs - fp32_probs)))

int8_sess = ort.InferenceSession(str(QUANT_ONNX), providers=["CPUExecutionProvider"])
int8_probs = softmax_np(int8_sess.run(None, ort_inputs)[0])
int8_diff = float(np.max(np.abs(pt_probs - int8_probs)))
label_agree = float(np.mean(np.argmax(pt_probs, axis=-1) == np.argmax(int8_probs, axis=-1)))

print(f"FP32 ONNX vs PyTorch max abs diff: {fp32_diff:.6f}")
print(f"INT8 ONNX vs PyTorch max abs diff: {int8_diff:.6f} | label agreement: {label_agree * 100:.1f}%")
assert fp32_diff < 1e-2, f"FP32 parity check failed: {fp32_diff}"
assert label_agree >= 0.96, f"INT8 label agreement too low: {label_agree}"
print("Gate C parity check PASSED.")

## 7. Upload to Hugging Face Hub (Human Step)
Set `HF_REPO_ID` and authenticate with a write token before running.


In [ ]:
HF_REPO_ID = "your-username/sajag-scam-classifier"  # <-- REPLACE WITH YOUR HF REPO
UPLOAD_FILES = ["model_quantized.onnx", "tokenizer.json", "config.json", "test_metrics.json"]

print("Run the commands below in Colab (or locally) after logging in via huggingface_hub:")
print(f"  hf auth login  # or huggingface-cli login")
for fname in UPLOAD_FILES:
    print(f"  hf upload {HF_REPO_ID} ./muril_onnx/{fname} {fname}")

# Uncomment to upload directly from Colab:
# from huggingface_hub import HfApi, login
# login()  # prompts for HF token
# api = HfApi()
# api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
# for fname in UPLOAD_FILES:
#     api.upload_file(
#         path_or_fileobj=str(ONNX_DIR / fname),
#         path_in_repo=fname,
#         repo_id=HF_REPO_ID,
#         repo_type="model",
#     )
# print("Uploaded all artifacts to", HF_REPO_ID)